# Motion-stratified robustness figure

Robustness-tier check (CLAUDE.md, "Motion stratification") on claim 2: does
connectome similarity depend on head motion, and does the within-task >
between-task ordering survive when motion is held down? Reads
`output_data/motion_strata/motion_strata.tsv` (written by `run-motion-strata`)
and plots only — no similarity computation here, which lives in
`analysis/motion_strata.py`.

**Standalone figure, deliberately not placed in `connectome_figure.svg`** —
the domain panels' placement there was an explicit, user-requested exception
and does not generalize (CLAUDE.md).

This notebook renders the single headline panel — the correlation
(similarity) result itself. The duration/tSNR balance audit and the
permutation-test effect sizes are still computed by `run-motion-strata` into
`motion_balance.tsv` and `motion_permutation.tsv`; they are reported as text
(key stats and ranges), not as figure panels here.

Bars are grouped with all within-task bins next to one another and all
between-task bins next to one another (task outer, motion-pairing inner), so
the large task effect reads as two clean blocks rather than interleaving with
the much smaller motion effect.

Panel:
1. `motion_bins.png` — the six low/low, low/high, high/high x within/between-task
   bins, one group of bars per network, "cell" split (median split within each
   (subject, dataset) cell — orthogonal to both by construction).


In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from airoh.figures import panel_size

FIGURE_DPI = int(os.environ.get("FIGURE_MONTAGE_DPI", 300))

output_dir = Path(os.environ.get("OUTPUT_DATA_DIR", "../output_data")).resolve()
figures_base = Path(os.environ.get("FIGURES_DIR", output_dir / "figures")).resolve()

project_root = output_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

figure_dir = figures_base / "figure_motion"
figure_dir.mkdir(parents=True, exist_ok=True)

with open(project_root / "invoke.yaml") as handle:
    invoke_config = yaml.safe_load(handle)

PARCELLATION = invoke_config["parcellation"]
NETWORK_ORDER = invoke_config["parcellations"][PARCELLATION]["network_order"]
MEASURE = invoke_config.get("analysis_measure", "pearson")

motion_dir = output_dir / "motion_strata"
motion_bins = pd.read_csv(motion_dir / "motion_strata.tsv", sep="\t")

print(f"📂 {PARCELLATION}, measure={MEASURE}: {len(motion_bins)} motion-bin rows")


📂 cneuromod2026, measure=pearson: 108 motion-bin rows


In [2]:
def save_legend(handles, labels, name, default_size, ncol=None, fontsize=7):
    """Render `handles`/`labels` alone into `{name}` as a horizontal strip."""
    if not handles:
        return
    figsize = panel_size(f"figure_motion/{name}", default_size)
    fig = plt.figure(figsize=figsize, layout="constrained")
    fig.legend(
        handles, labels, loc="center",
        ncol=ncol or min(len(handles), 5), fontsize=fontsize, frameon=False,
    )
    fig.savefig(figure_dir / name, dpi=FIGURE_DPI)
    plt.close(fig)
    print(f"✅ wrote {figure_dir / name} at {figsize} in")


In [3]:
# Shared color scheme (dataviz skill, "color-formula"): hue = task (categorical,
# 2 series, validated pair — blue/orange, worst adjacent CVD dE 24.7 light), alpha
# = motion pairing (ordinal: low-low -> low-high -> high-high increases apparent
# motion contrast, so darker/more opaque = more motion mismatch). This keeps every
# panel on the same two-hue system instead of six unrelated flat colors.
TASK_HUE = {"within-task": "#2a78d6", "between-task": "#eb6834"}
MOTION_ALPHA = {"low-low": 0.40, "low-high": 0.70, "high-high": 1.0}
MOTION_ORDER = ["low-low", "low-high", "high-high"]
TASK_ORDER = ["within-task", "between-task"]
# Task outer, motion inner: groups all within-task bars together and all
# between-task bars together, so the (much larger) task effect reads as two
# clean blocks instead of interleaving with the (tiny) motion effect.
BIN_ORDER = [f"{m}/{t}" for t in TASK_ORDER for m in MOTION_ORDER]


def bin_color(bin_label):
    motion, task = bin_label.split("/")
    return TASK_HUE[task], MOTION_ALPHA[motion]

In [4]:
# Panel 1 — motion_bins.png: six motion x task bins, "cell" split, all networks.
# Hue marks task (within/between), alpha marks motion pairing — so the panel
# itself shows the finding: within a hue, the three alpha steps barely move,
# because the motion effect is tiny next to the task effect.
figsize = panel_size("figure_motion/motion_bins.png", (6.0, 4.5))
fig, ax = plt.subplots(figsize=figsize, layout="constrained")

cell_bins = motion_bins[motion_bins["split"] == "cell"]

x = np.arange(len(NETWORK_ORDER))
width = 0.13
legend_handles, legend_labels = [], []
if len(cell_bins):
    for i, bin_label in enumerate(BIN_ORDER):
        color, alpha = bin_color(bin_label)
        values = []
        for network in NETWORK_ORDER:
            row = cell_bins[(cell_bins["network"] == network) & (cell_bins["bin"] == bin_label)]
            values.append(row["median"].iloc[0] if len(row) else np.nan)
        bars = ax.bar(x + (i - 2.5) * width, values, width, color=color, alpha=alpha,
                       edgecolor="white", linewidth=0.4)
        legend_handles.append(bars[0])
        legend_labels.append(bin_label)
    ax.set_xticks(x)
    ax.set_xticklabels(NETWORK_ORDER, rotation=45, ha="right", fontsize=7)
    ax.set_ylabel("median similarity (Fisher-z)")
    ax.spines[["top", "right"]].set_visible(False)
else:
    ax.text(0.5, 0.5, "no QC-covered sessions\n(smoke run, or too few cells)",
            ha="center", va="center", transform=ax.transAxes, color="0.5", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])

fig.savefig(figure_dir / "motion_bins.png", dpi=FIGURE_DPI)
plt.close(fig)
print(f"✅ wrote {figure_dir / 'motion_bins.png'} at {figsize} in")

save_legend(legend_handles, legend_labels, "motion_bins_legend.png", (6.0, 0.9), ncol=3)


✅ wrote /home/pbellec/git/cneuromod.all.connectome_stats/output_data/figures/figure_motion/motion_bins.png at (6.0, 4.5) in


✅ wrote /home/pbellec/git/cneuromod.all.connectome_stats/output_data/figures/figure_motion/motion_bins_legend.png at (6.0, 0.9) in
